# RAPID2 — Exploratory Data Analysis (EDA)

**Purpose**

This notebook establishes a reusable, decision-oriented baseline EDA for the RAPID2 production data source.

It is designed to answer the initial project questions:

- What jurisdictions and populations are available?
- What is the size, coverage and time range of the source?
- What does one RAPID2 record represent?
- Which fields are complete, sparse, unique or duplicated?
- What are the dominant categories, statuses and taxonomies?
- How has the population changed over time?
- Are there data-quality or structural anomalies?
- What fields are potentially useful for downstream ontology, search, analytics or GenAI use cases?
- What should be investigated further before solution design?

> **Important:** RAPID2 is live production data. The notebook is read-only and uses aggregation wherever possible. It does not modify source data.

## 1. Working principle

The existing `ia_data_collation` notebook is the reference for connectivity and RAPID2 extraction.

This notebook intentionally **reuses the same BigQuery client pattern that is already working** rather than introducing a new authentication/query mechanism.

Jurisdiction handling is deliberately separated from the EDA:

**Discover jurisdictions → review population → select an optional jurisdiction → run detailed EDA.**

Do not hard-code a jurisdiction into individual SQL statements.

In [ ]:
# ============================================================
# 2. IMPORTS
# ============================================================

import os
import warnings
import pandas as pd
from IPython.display import display
from google.cloud import bigquery

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

In [ ]:
# ============================================================
# 3. CONFIGURATION
#     Same working configuration as ia_data_collation.
# ============================================================

ANALYTICS_PROJECT = "hsbc-12211200-cmlpwb-prod"
DATA_PROJECT = "hsbc-11545401-cmpdatcwrs1-prod"
DATASET = "rc_curated_prod"
TABLE = "aa_hsbc_rapid2_hzn_record"

LOCATION = "europe-west2"

os.environ["GOOGLE_CLOUD_PROJECT"] = ANALYTICS_PROJECT
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
    "/home/jovyan/.config/gcloud/application_default_credentials.json"
)

TABLE_FQN = f"`{DATA_PROJECT}.{DATASET}.{TABLE}`"
TABLE_ID = f"{DATA_PROJECT}.{DATASET}.{TABLE}"

# RAPID2 visualisation is currently approved for these jurisdictions.
APPROVED_JURISDICTIONS = [
    "United States of America",
    "United Kingdom (Ring-Fenced Bank)",
    "Hong Kong (HSBC)",
]

# Start with None. Select only AFTER the discovery output is reviewed.
SELECTED_JURISDICTION = None

# Optional filters. Keep None for the baseline population.
CREATED_FROM = None       # e.g. "2020-01-01"
CREATED_TO = None         # e.g. "2025-12-31"
FILTER_ACTIVE_ONLY = False
FILTER_INGEST_CHANNEL = None
FILTER_RECORD_CATEGORY = None

In [ ]:
# ============================================================
# 4. BIGQUERY CLIENT
#     Deliberately mirrors the working ia_data_collation code.
# ============================================================

bq_client = bigquery.Client(
    project=ANALYTICS_PROJECT,
    location=LOCATION,
)

print("Analytics project :", ANALYTICS_PROJECT)
print("Data project      :", DATA_PROJECT)
print("Dataset           :", DATASET)
print("Table             :", TABLE)
print("Location          :", LOCATION)

In [ ]:
# ============================================================
# 5. CONNECTION / TABLE ACCESS TEST
# ============================================================

# Use get_table() as the first test.
# This avoids INFORMATION_SCHEMA as the first connectivity check.

table_ref = bq_client.get_table(TABLE_ID)

print("Connection / access: OK")
print("Full table ID      :", table_ref.full_table_id)
print("Rows               :", f"{table_ref.num_rows:,}")
print("Size (GB)          :", round(table_ref.num_bytes / (1024**3), 2))
print("Created            :", table_ref.created)
print("Modified           :", table_ref.modified)
print("Physical columns   :", len(table_ref.schema))

## 6. Physical schema

Inspect the physical schema before querying values. This prevents assumptions about field names, types and nullability and provides the basis for the generic EDA functions below.

In [ ]:
# ============================================================
# 6A. SCHEMA INVENTORY
# ============================================================

schema_rows = []

for field in table_ref.schema:
    schema_rows.append({
        "POSITION": len(schema_rows) + 1,
        "FIELD_NAME": field.name,
        "DATA_TYPE": field.field_type,
        "MODE": field.mode,
        "DESCRIPTION": field.description,
    })

schema_df = pd.DataFrame(schema_rows)

display(schema_df)
print(f"Physical columns: {len(schema_df)}")

## 7. Jurisdiction discovery — run this before selecting a jurisdiction

This is the first substantive RAPID2 analysis.

The output shows all jurisdictions present in the source and their population. The approved jurisdiction list is shown alongside the result so that downstream analysis does not accidentally expose an unapproved population.

**Decision point:** review this table, then populate `SELECTED_JURISDICTION` if a jurisdiction-specific EDA is required.

In [ ]:
# ============================================================
# 7A. DISCOVER JURISDICTIONS
# ============================================================

jurisdiction_sql = f"""
SELECT
    JRIS_CDE AS JURISDICTION_CODE,
    COUNT(*) AS RECORD_COUNT,
    COUNT(DISTINCT RECORD_ID) AS DISTINCT_RECORD_COUNT,
    MIN(CREATED_ON_DTM) AS FIRST_CREATED,
    MAX(CREATED_ON_DTM) AS LAST_CREATED,
    COUNTIF(IS_ACTIVE_IND = 1) AS ACTIVE_RECORD_COUNT
FROM {TABLE_FQN}
GROUP BY JRIS_CDE
ORDER BY RECORD_COUNT DESC
"""

jurisdiction_df = (
    bq_client.query(jurisdiction_sql)
    .result()
    .to_dataframe()
)

jurisdiction_df["APPROVED_FOR_VISUALISATION"] = (
    jurisdiction_df["JURISDICTION_CODE"].isin(APPROVED_JURISDICTIONS)
)

display(jurisdiction_df)

In [ ]:
# ============================================================
# 7B. APPROVED JURISDICTION SUMMARY
# ============================================================

approved_df = jurisdiction_df[
    jurisdiction_df["APPROVED_FOR_VISUALISATION"]
].copy()

display(approved_df)

print("Approved jurisdictions found:")
for j in approved_df["JURISDICTION_CODE"].tolist():
    print(" -", j)

### 7C. Build the central EDA filter

The filter is applied centrally to every subsequent EDA query. This prevents hard-coded jurisdiction conditions from being scattered across the notebook.

In [ ]:
# ============================================================
# 7C. BUILD CENTRAL EDA FILTER
# ============================================================

where_clauses = []

if SELECTED_JURISDICTION:
    if SELECTED_JURISDICTION not in APPROVED_JURISDICTIONS:
        raise ValueError(
            f"'{SELECTED_JURISDICTION}' is not in the approved jurisdiction list."
        )
    value = SELECTED_JURISDICTION.replace("'", "''")
    where_clauses.append(f"JRIS_CDE = '{value}'")

if CREATED_FROM:
    where_clauses.append(
        f"DATE(CREATED_ON_DTM) >= '{CREATED_FROM}'"
    )

if CREATED_TO:
    where_clauses.append(
        f"DATE(CREATED_ON_DTM) <= '{CREATED_TO}'"
    )

if FILTER_ACTIVE_ONLY:
    where_clauses.append("IS_ACTIVE_IND = 1")

if FILTER_INGEST_CHANNEL:
    value = FILTER_INGEST_CHANNEL.replace("'", "''")
    where_clauses.append(f"INGEST_CHNL_ID = '{value}'")

if FILTER_RECORD_CATEGORY:
    value = FILTER_RECORD_CATEGORY.replace("'", "''")
    where_clauses.append(f"RECORD_CAT_CDE = '{value}'")

WHERE_SQL = (
    "\nWHERE " + "\n  AND ".join(where_clauses)
    if where_clauses
    else ""
)

print("EDA population filter:")
print(WHERE_SQL if WHERE_SQL else "FULL RAPID2 POPULATION")

## 8. Base population profile

This establishes the basic unit of analysis and immediately highlights whether `RECORD_ID` and `ORIGIN_ALERT_ID` behave as expected.

In [ ]:
# ============================================================
# 8. BASE POPULATION PROFILE
# ============================================================

population_sql = f"""
SELECT
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT RECORD_ID) AS DISTINCT_RECORD_ID,
    COUNT(DISTINCT ORIGIN_ALERT_ID) AS DISTINCT_ORIGIN_ALERT_ID,
    COUNTIF(RECORD_ID IS NULL) AS NULL_RECORD_ID,
    COUNTIF(ORIGIN_ALERT_ID IS NULL) AS NULL_ORIGIN_ALERT_ID,
    COUNTIF(IS_ACTIVE_IND = 1) AS ACTIVE_RECORDS,
    COUNTIF(IS_ACTIVE_IND = 0) AS INACTIVE_RECORDS,
    MIN(CREATED_ON_DTM) AS MIN_CREATED_ON_DTM,
    MAX(CREATED_ON_DTM) AS MAX_CREATED_ON_DTM,
    MIN(UPDATED_ON_DTM) AS MIN_UPDATED_ON_DTM,
    MAX(UPDATED_ON_DTM) AS MAX_UPDATED_ON_DTM
FROM {TABLE_FQN}
{WHERE_SQL}
"""

population_df = (
    bq_client.query(population_sql)
    .result()
    .to_dataframe()
)

display(population_df)

## 9. Completeness / null analysis

Null analysis is critical because downstream ontology, retrieval and GenAI design depend heavily on field availability.

The query is generated from the physical schema and runs as one BigQuery aggregation rather than one query per field.

In [ ]:
# ============================================================
# 9. COLUMN COMPLETENESS
# ============================================================

simple_fields = [
    f for f in table_ref.schema
    if f.mode != "REPEATED" and f.field_type != "RECORD"
]

null_exprs = [
    f"COUNTIF(`{f.name}` IS NULL) AS `{f.name}__NULL_COUNT`"
    for f in simple_fields
]

if null_exprs:
    completeness_sql = f"""
    SELECT
        COUNT(*) AS TOTAL_ROWS,
        {", ".join(null_exprs)}
    FROM {TABLE_FQN}
    {WHERE_SQL}
    """

    completeness_wide = (
        bq_client.query(completeness_sql)
        .result()
        .to_dataframe()
    )

    total_rows = int(completeness_wide.loc[0, "TOTAL_ROWS"])
    completeness_rows = []

    for f in simple_fields:
        null_count = int(
            completeness_wide.loc[0, f"{f.name}__NULL_COUNT"]
        )
        completeness_rows.append({
            "FIELD_NAME": f.name,
            "DATA_TYPE": f.field_type,
            "NULL_COUNT": null_count,
            "NON_NULL_COUNT": total_rows - null_count,
            "NULL_PCT": round(
                100 * null_count / total_rows, 2
            ) if total_rows else None,
            "COMPLETENESS_PCT": round(
                100 * (total_rows - null_count) / total_rows, 2
            ) if total_rows else None,
        })

    completeness_df = (
        pd.DataFrame(completeness_rows)
        .sort_values(
            ["NULL_PCT", "FIELD_NAME"],
            ascending=[False, True]
        )
        .reset_index(drop=True)
    )

    display(completeness_df)
else:
    print("No simple fields found.")

## 10. Approximate cardinality

Cardinality distinguishes identifiers, low-cardinality dimensions, taxonomy fields and potentially high-value analytical attributes.

`APPROX_COUNT_DISTINCT` is used to keep the operation efficient on a production-sized table.

In [ ]:
# ============================================================
# 10. APPROXIMATE CARDINALITY
# ============================================================

cardinality_fields = [
    f for f in table_ref.schema
    if f.mode != "REPEATED" and f.field_type != "RECORD"
]

cardinality_exprs = [
    f"APPROX_COUNT_DISTINCT(`{f.name}`) AS `{f.name}`"
    for f in cardinality_fields
]

cardinality_sql = f"""
SELECT
    {", ".join(cardinality_exprs)}
FROM {TABLE_FQN}
{WHERE_SQL}
"""

cardinality_wide = (
    bq_client.query(cardinality_sql)
    .result()
    .to_dataframe()
)

cardinality_rows = []

for f in cardinality_fields:
    value = cardinality_wide.loc[0, f.name]
    cardinality_rows.append({
        "FIELD_NAME": f.name,
        "DATA_TYPE": f.field_type,
        "APPROX_DISTINCT_VALUES": int(value) if pd.notna(value) else 0,
    })

cardinality_df = (
    pd.DataFrame(cardinality_rows)
    .sort_values("APPROX_DISTINCT_VALUES", ascending=False)
    .reset_index(drop=True)
)

display(cardinality_df)

## 11. Duplicate / key integrity analysis

RAPID2 is expected to contain regulatory/alert records. We therefore test likely identifiers independently rather than assuming that every physical row is one unique business object.

In [ ]:
# ============================================================
# 11. KEY INTEGRITY
# ============================================================

key_sql = f"""
SELECT
    COUNT(*) AS ROWS,
    COUNT(DISTINCT RECORD_ID) AS DISTINCT_RECORD_ID,
    COUNT(DISTINCT ORIGIN_ALERT_ID) AS DISTINCT_ORIGIN_ALERT_ID,
    COUNTIF(RECORD_ID IS NULL) AS NULL_RECORD_ID,
    COUNTIF(ORIGIN_ALERT_ID IS NULL) AS NULL_ORIGIN_ALERT_ID
FROM {TABLE_FQN}
{WHERE_SQL}
"""

key_df = bq_client.query(key_sql).result().to_dataframe()
display(key_df)

In [ ]:
# ============================================================
# 11A. DUPLICATE RECORD_ID GROUPS
# ============================================================

duplicate_record_sql = f"""
SELECT
    RECORD_ID,
    COUNT(*) AS ROW_COUNT,
    MIN(CREATED_ON_DTM) AS FIRST_CREATED,
    MAX(UPDATED_ON_DTM) AS LAST_UPDATED
FROM {TABLE_FQN}
{WHERE_SQL}
{"AND" if WHERE_SQL else "WHERE"} RECORD_ID IS NOT NULL
GROUP BY RECORD_ID
HAVING COUNT(*) > 1
ORDER BY ROW_COUNT DESC
LIMIT 100
"""

duplicate_record_df = (
    bq_client.query(duplicate_record_sql)
    .result()
    .to_dataframe()
)

display(duplicate_record_df)
print("Displayed maximum: 100 duplicate RECORD_ID groups.")

## 12. Core categorical distributions

These dimensions are especially relevant for initial project understanding and future ontology mapping.

In [ ]:
# ============================================================
# 12. CORE CATEGORICAL DISTRIBUTIONS
# ============================================================

CORE_CATEGORICAL_FIELDS = [
    "JRIS_CDE",
    "RECORD_CAT_CDE",
    "STAT_CDE",
    "IS_ACTIVE_IND",
    "INGEST_RULE_TYPE",
    "RGLT_BDY_CDE",
    "RISK_STWRD_AREA_CDE",
]

def categorical_profile(field_name, limit=50):
    sql = f"""
    SELECT
        `{field_name}` AS VALUE,
        COUNT(*) AS RECORD_COUNT,
        ROUND(
            100 * SAFE_DIVIDE(
                COUNT(*),
                SUM(COUNT(*)) OVER ()
            ),
            2
        ) AS PCT_OF_POPULATION
    FROM {TABLE_FQN}
    {WHERE_SQL}
    GROUP BY `{field_name}`
    ORDER BY RECORD_COUNT DESC
    LIMIT {limit}
    """

    return (
        bq_client.query(sql)
        .result()
        .to_dataframe()
    )

for field in CORE_CATEGORICAL_FIELDS:
    if field in schema_df["FIELD_NAME"].values:
        print(f"\n### {field}")
        display(categorical_profile(field))

## 13. Taxonomy / theme distribution

RAPID2 contains regulatory themes and risk-taxonomy attributes. These are important candidates for future ontology mapping and for connections to Reg Map / Helios.

In [ ]:
# ============================================================
# 13. TAXONOMY / THEME DISTRIBUTIONS
# ============================================================

TAXONOMY_FIELDS = [
    "RISK_TXNMY_LVL_1",
    "RISK_TXNMY_LVL_2",
    "RISK_TXNMY_LVL_3",
    "RECORD_THEME_CODE",
]

for field in TAXONOMY_FIELDS:
    if field in schema_df["FIELD_NAME"].values:
        print(f"\n### {field}")
        display(categorical_profile(field, limit=50))

## 14. Temporal EDA

The objective is to understand whether the source is continuously populated, seasonal, affected by ingestion gaps, growing/shrinking or dominated by particular periods.

Monthly aggregation is preferable initially to extracting the full date population into pandas.

In [ ]:
# ============================================================
# 14. MONTHLY RECORD DISTRIBUTION
# ============================================================

monthly_sql = f"""
SELECT
    DATE_TRUNC(DATE(CREATED_ON_DTM), MONTH) AS CREATED_MONTH,
    COUNT(*) AS RECORD_COUNT,
    COUNT(DISTINCT RECORD_ID) AS DISTINCT_RECORD_COUNT,
    COUNTIF(IS_ACTIVE_IND = 1) AS ACTIVE_RECORD_COUNT
FROM {TABLE_FQN}
{WHERE_SQL}
GROUP BY CREATED_MONTH
ORDER BY CREATED_MONTH
"""

monthly_df = (
    bq_client.query(monthly_sql)
    .result()
    .to_dataframe()
)

display(monthly_df)

In [ ]:
# ============================================================
# 14A. RECORD AGE / UPDATE LAG
# ============================================================

age_sql = f"""
SELECT
    APPROX_QUANTILES(
        DATE_DIFF(
            DATE(COALESCE(UPDATED_ON_DTM, CREATED_ON_DTM)),
            DATE(CREATED_ON_DTM),
            DAY
        ),
        10
    ) AS RECORD_LIFETIME_DAYS_DECILES,

    APPROX_QUANTILES(
        DATE_DIFF(
            CURRENT_DATE(),
            DATE(CREATED_ON_DTM),
            DAY
        ),
        10
    ) AS CURRENT_AGE_DAYS_DECILES
FROM {TABLE_FQN}
{WHERE_SQL}
"""

age_df = bq_client.query(age_sql).result().to_dataframe()
display(age_df)

## 15. Data freshness / stale records

This helps establish whether RAPID2 behaves as a live operational source or primarily as a historical repository.

In [ ]:
# ============================================================
# 15. FRESHNESS PROFILE
# ============================================================

freshness_sql = f"""
SELECT
    COUNT(*) AS TOTAL_RECORDS,

    COUNTIF(
        DATE_DIFF(CURRENT_DATE(), DATE(UPDATED_ON_DTM), DAY) <= 30
    ) AS UPDATED_LAST_30_DAYS,

    COUNTIF(
        DATE_DIFF(CURRENT_DATE(), DATE(UPDATED_ON_DTM), DAY) BETWEEN 31 AND 90
    ) AS UPDATED_31_TO_90_DAYS,

    COUNTIF(
        DATE_DIFF(CURRENT_DATE(), DATE(UPDATED_ON_DTM), DAY) > 90
    ) AS NOT_UPDATED_FOR_OVER_90_DAYS,

    COUNTIF(UPDATED_ON_DTM IS NULL) AS NULL_UPDATED_DATE
FROM {TABLE_FQN}
{WHERE_SQL}
"""

freshness_df = (
    bq_client.query(freshness_sql)
    .result()
    .to_dataframe()
)

display(freshness_df)

## 16. Relationship / linkage readiness

The objective here is not to assume the final ontology. It is to measure whether RAPID2 contains stable fields that could potentially connect to other sources.

In [ ]:
# ============================================================
# 16. CANDIDATE LINKAGE FIELDS
# ============================================================

CANDIDATE_LINK_FIELDS = [
    "RECORD_ID",
    "ORIGIN_ALERT_ID",
    "JRIS_CDE",
    "RECORD_CAT_CDE",
    "RGLT_BDY_CDE",
    "RISK_STWRD_AREA_CDE",
    "RECORD_THEME_CODE",
    "TITLE",
    "URL_TEXT",
]

available_link_fields = [
    f for f in CANDIDATE_LINK_FIELDS
    if f in schema_df["FIELD_NAME"].values
]

link_exprs = []

for field in available_link_fields:
    link_exprs.append(f"""
        COUNT(`{field}`) AS `{field}__NON_NULL`,
        APPROX_COUNT_DISTINCT(`{field}`) AS `{field}__APPROX_DISTINCT`
    """)

link_sql = f"""
SELECT
    {", ".join(link_exprs)}
FROM {TABLE_FQN}
{WHERE_SQL}
"""

link_wide = (
    bq_client.query(link_sql)
    .result()
    .to_dataframe()
)

link_rows = []

for field in available_link_fields:
    non_null = int(link_wide.loc[0, f"{field}__NON_NULL"])
    distinct = int(link_wide.loc[0, f"{field}__APPROX_DISTINCT"])

    link_rows.append({
        "FIELD_NAME": field,
        "NON_NULL_COUNT": non_null,
        "APPROX_DISTINCT": distinct,
        "APPROX_UNIQUENESS_PCT": round(
            100 * distinct / non_null, 2
        ) if non_null else None,
    })

link_df = pd.DataFrame(link_rows).sort_values(
    "APPROX_UNIQUENESS_PCT",
    ascending=False
)

display(link_df)

## 17. Text-field profiling

RAPID2 contains fields such as title, summary/content and introductory text. Before deciding whether these fields are suitable for search, embeddings, LLM summarisation or semantic matching, measure their basic characteristics.

The notebook computes aggregate text statistics in BigQuery rather than downloading the full text population.

In [ ]:
# ============================================================
# 17. TEXT FIELD PROFILE
# ============================================================

TEXT_FIELDS = [
    "TITLE",
    "SUMM_CONTENT",
    "SUMM_UPDT",
    "INTRO_TEXT",
    "URL_TEXT",
]

available_text_fields = [
    f for f in TEXT_FIELDS
    if f in schema_df["FIELD_NAME"].values
]

text_exprs = []

for field in available_text_fields:
    text_exprs.append(f"""
        COUNT(`{field}`) AS `{field}__NON_NULL`,
        COUNTIF(TRIM(CAST(`{field}` AS STRING)) = '') AS `{field}__EMPTY`,
        APPROX_QUANTILES(
            LENGTH(CAST(`{field}` AS STRING)),
            10
        ) AS `{field}__LENGTH_DECILES`
    """)

if text_exprs:
    text_sql = f"""
    SELECT
        {", ".join(text_exprs)}
    FROM {TABLE_FQN}
    {WHERE_SQL}
    """

    text_wide = (
        bq_client.query(text_sql)
        .result()
        .to_dataframe()
    )

    text_rows = []

    for field in available_text_fields:
        deciles = text_wide.loc[0, f"{field}__LENGTH_DECILES"]

        text_rows.append({
            "FIELD_NAME": field,
            "NON_NULL_COUNT": text_wide.loc[0, f"{field}__NON_NULL"],
            "EMPTY_COUNT": text_wide.loc[0, f"{field}__EMPTY"],
            "LENGTH_P10": deciles[1] if deciles else None,
            "LENGTH_P50": deciles[5] if deciles else None,
            "LENGTH_P90": deciles[9] if deciles else None,
        })

    text_profile_df = pd.DataFrame(text_rows)
    display(text_profile_df)
else:
    print("No configured text fields were found in the physical schema.")

## 18. Data-quality anomaly checks

These are high-value structural checks that can expose issues before modelling or source integration.

In [ ]:
# ============================================================
# 18. DATA-QUALITY ANOMALIES
# ============================================================

anomaly_sql = f"""
SELECT
    COUNTIF(RECORD_ID IS NULL) AS NULL_RECORD_ID,

    COUNTIF(
        CREATED_ON_DTM IS NULL
    ) AS NULL_CREATED_DATE,

    COUNTIF(
        UPDATED_ON_DTM < CREATED_ON_DTM
    ) AS UPDATED_BEFORE_CREATED,

    COUNTIF(
        IS_ACTIVE_IND NOT IN (0, 1)
        AND IS_ACTIVE_IND IS NOT NULL
    ) AS INVALID_ACTIVE_INDICATOR,

    COUNTIF(
        TRIM(CAST(TITLE AS STRING)) = ''
    ) AS EMPTY_TITLE,

    COUNTIF(
        URL_TEXT IS NOT NULL
        AND NOT STARTS_WITH(
            LOWER(TRIM(CAST(URL_TEXT AS STRING))),
            'http'
        )
    ) AS NON_HTTP_URL,

    COUNTIF(
        JRIS_CDE IS NULL
    ) AS NULL_JURISDICTION
FROM {TABLE_FQN}
{WHERE_SQL}
"""

anomaly_df = (
    bq_client.query(anomaly_sql)
    .result()
    .to_dataframe()
)

display(anomaly_df)

## 19. High-level source segmentation

This combines the most important dimensions into one compact view and is useful for initial project discussion.

In [ ]:
# ============================================================
# 19. SOURCE SEGMENTATION
# ============================================================

segmentation_sql = f"""
SELECT
    JRIS_CDE AS JURISDICTION,
    RECORD_CAT_CDE AS RECORD_CATEGORY,
    IS_ACTIVE_IND AS ACTIVE_INDICATOR,
    COUNT(*) AS RECORD_COUNT,
    COUNT(DISTINCT RECORD_ID) AS DISTINCT_RECORD_COUNT
FROM {TABLE_FQN}
{WHERE_SQL}
GROUP BY
    JURISDICTION,
    RECORD_CATEGORY,
    ACTIVE_INDICATOR
ORDER BY RECORD_COUNT DESC
"""

segmentation_df = (
    bq_client.query(segmentation_sql)
    .result()
    .to_dataframe()
)

display(segmentation_df.head(100))

# 20. Project questions answered by this EDA

| Question | Primary section |
|---|---|
| Which jurisdictions exist? | 7 |
| Which jurisdictions are approved? | 7 |
| How large is RAPID2? | 5, 8 |
| What is the time coverage? | 8, 14 |
| What does a record look like? | 6, 8 |
| Are RECORD_ID / ALERT_ID unique? | 8, 11 |
| Which fields are incomplete? | 9 |
| Which fields have useful cardinality? | 10 |
| What are the dominant categories/statuses? | 12 |
| What is the taxonomy/theme structure? | 13 |
| Is the source continuously populated? | 14 |
| How fresh is the data? | 15 |
| Which fields could support source linkage? | 16 |
| Is the text suitable for downstream GenAI/search analysis? | 17 |
| Are there obvious data-quality problems? | 18 |
| How is the population segmented? | 19 |

# 21. Recommended next EDA layer — cross-source analysis

Once RAPID2 baseline EDA is complete, the next high-value analysis should be **relationship discovery across RAPID2 → Reg Map → Helios**, rather than simply adding more generic charts.

Recommended next steps:

1. Identify candidate keys and semantic join paths.
2. Measure RAPID2-to-Reg Map match rates.
3. Measure whether regulatory themes / risk taxonomy can bridge into RGL/RSM/control structures.
4. Measure whether Reg Map controls connect to Helios obligations.
5. Quantify unmatched and ambiguous records.
6. Identify one-to-many and many-to-many relationships.
7. Establish which fields are authoritative in each source.

This will provide the evidence required for the ontology and target-solution architecture.

## 22. Execution notes

- Run Sections **1–7** first.
- Confirm the jurisdiction universe.
- Leave `SELECTED_JURISDICTION = None` for the baseline population, or select one approved jurisdiction.
- Then run Sections **7C onward**.
- Keep the existing working BigQuery authentication pattern.
- Prefer BigQuery aggregation and approximate statistics to moving raw production data into pandas.
- Apply jurisdiction/date filters before any future row-level extraction.
- This notebook is read-only.